In [ ]:
# This notebook is to streamline the process from energy model to free energy model

## Importing required libraries

In [2]:
import os
import pickle
import catboost as cb
from time import time
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
# from rdkit import Chem
# from rdkit.Chem import AllChem

c:\Users\USER\anaconda3\lib\site-packages\xgboost\compat.py:85: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


## Adding fingerprint

In [3]:
df=pd.read_excel(r"C:\Users\USER\Downloads\9237 Ea predict dataset.xlsx")
smilesr1=df["Reactant 1"]
smilesr2=df["Reactant 2"]
smilesp1=df["Product 1"]
smilesp2=df["Product 2"]
def generateDenseECFP(smiles_arr,featureName,radius,nBits):
    list=[]
    for i in range (len(smiles_arr)):  
        mol = Chem.MolFromSmiles(smiles_arr[i])
        ecfp = AllChem.GetMorganFingerprintAsBitVect(mol,radius,nBits)
        ecfp=ecfp.ToBitString()
        list.append(ecfp)

    ECFP=pd.DataFrame({featureName:list})
    helper_df=ECFP[featureName].str.split('', expand=True)

    for i in range(1,nBits+1):
    #for i in range(nBits):
        helper_df[i]=helper_df[i].astype(int)

    helper_df = helper_df.add_prefix(featureName)

   # helper_df = helper_df.loc[:, (helper_df != 0).any(axis=0)]
    helper_df.drop(columns=[helper_df.columns[-1]], inplace=True)
    helper_df.drop(columns=[helper_df.columns[0]], inplace=True)
    return helper_df

# function to convert to subscript 
def get_sub(x): 
	normal = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+-=()"
	sub_s = "ₐ₈CDₑբGₕᵢⱼₖₗₘₙₒₚQᵣₛₜᵤᵥwₓᵧZₐ♭꜀ᑯₑբ₉ₕᵢⱼₖₗₘₙₒₚ૧ᵣₛₜᵤᵥwₓᵧ₂₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎"
	res = x.maketrans(''.join(normal), ''.join(sub_s)) 
	return x.translate(res) 
ECFPr1=generateDenseECFP(smilesr1,"Reactant{}".format(get_sub('1')),3,1024)
ECFPr2=generateDenseECFP(smilesr2,"Reactant{}".format(get_sub('2')),3,1024)
ECFPp1=generateDenseECFP(smilesp1,"Product{}".format(get_sub('1')),3,1024)
ECFPp2=generateDenseECFP(smilesp2,"Product{}".format(get_sub('2')),3,1024)

result = pd.concat([df,ECFPp1,ECFPp2,ECFPr1,ECFPr2], axis=1, join='inner')

result.to_csv(r"C:\Users\USER\Downloads\9237 Ea predict datasetwithfnp.csv", index=False)

[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not removing hydrogen atom without neighbors
[16:33:22] WARNING: not r

## Predicting Ea

In [12]:
modelEa=pickle.load(open(r"Ea.pkl",'rb'))
data_predict=pd.read_csv(r"C:\Users\USER\Downloads\9237 Ea predict datasetwithfnp.csv")
data_predict=data_predict.rename(columns={"energy": "Del E"})
data_153=data_predict[['Del E', 'nC(R1)', 'nC(R2)', 'nC(P2)', 'nO(R1)', 'nO(R2)', 'nO(P1)',
       'nO(P2)', 'nH(R1)', 'nH(R2)', 'nH(P2)', 'Cn-H', 'C-O', 'C-C',
       'numCO_Hbond', 'numC_OHbond', 'MolWt_R1', 'MolWt_P1', 'num_terC_R1',
       'num_terC_R2', 'num_terC_P1', 'num_terC_P2', 'num_secC_R1',
       'num_secC_R2', 'num_secC_P1', 'num_secC_P2', 'num_priC_R1',
       'num_priC_R2', 'num_priC_P1', 'num_priC_P2','Reactant₁712','Reactant₂712','Product₁712','Product₂712','Reactant₁795','Reactant₂795','Product₁795','Product₂795','Reactant₁474','Reactant₂474','Product₁474','Product₂474','Reactant₁1005','Reactant₂1005','Product₁1005','Product₂1005',]]
Ea=modelEa.predict(data_153)
Ea_df=pd.DataFrame()
Ea_df["Ea"]=Ea
predictedEawithreaction=pd.concat([data_predict[['reaction', 'Del E', 'Reactant 1', 'Reactant 2',
       'Product 1', 'Product 2','nC(R1)', 'nC(R2)', 'nC(P2)', 'nO(R1)', 'nO(R2)', 'nO(P1)',
       'nO(P2)', 'nH(R1)', 'nH(R2)', 'nH(P2)', 'Cn-H', 'C-O', 'C-C',
       'numCO_Hbond', 'numC_OHbond', 'MolWt_R1', 'MolWt_P1', 'num_terC_R1',
       'num_terC_R2', 'num_terC_P1', 'num_terC_P2', 'num_secC_R1',
       'num_secC_R2', 'num_secC_P1', 'num_priC_R1', 'num_priC_R2',
       'num_priC_P1', 'num_priC_P2']],Ea_df],axis=1, join='inner')
predictedEawithreaction.to_csv("predicted9k_1Dec.csv")

## Predicting G

In [3]:
data=pd.read_csv(r"D:\projects\activation_energy_prediction\ML\predicted9k_1Dec.csv")

In [4]:
data_153=data[['Del E','Ea', 'nC(R1)', 'nC(R2)', 'nC(P2)', 'nO(R1)', 'nO(R2)', 'nO(P1)',
       'nO(P2)', 'nH(R1)', 'nH(R2)', 'nH(P2)', 'Cn-H', 'C-O', 'C-C',
       'numCO_Hbond', 'numC_OHbond', 'MolWt_R1', 'MolWt_P1', 'num_terC_R1',
       'num_terC_R2', 'num_terC_P1', 'num_terC_P2', 'num_secC_R1',
       'num_secC_R2', 'num_secC_P1', 'num_priC_R1', 'num_priC_R2',
       'num_priC_P1', 'num_priC_P2']]

In [5]:
modelGa=pickle.load(open("D:\projects\\activation_energy_prediction\G_models\Ga19oct.pkl",'rb'))
Ga=pd.DataFrame()

In [6]:
x_predicGa=data_153.drop("Del E",axis=1)
for i in np.arange(300,950,50):
    x_predicGa["Tempearture"+str(i)]=i
    first_column = x_predicGa.pop('Tempearture'+str(i))
    x_predicGa.insert(0, 'Tempearture'+str(i), first_column)
    Ga["Ga"+str(i)]=modelGa.predict(x_predicGa).tolist()
    x_predicGa.drop("Tempearture"+str(i),axis=1,inplace=True)

In [7]:
for i in np.arange(300,950,50):
    for index in Ga.index:
        Ga["Ga"+str(i)][index]=str(Ga["Ga"+str(i)][index])
        Ga["Ga"+str(i)][index]=Ga["Ga"+str(i)][index].replace(']','')
        Ga["Ga"+str(i)][index]=Ga["Ga"+str(i)][index].replace('[','') 


In [ ]:
Ga_data=pd.concat([data[['reaction', 'Del E','Ea','Reactant 1', 'Reactant 2',
       'Product 1', 'Product 2']],Ga],axis=1, join='inner')
G_data=pd.concat([data[['reaction', 'Del E','Ea','Reactant 1', 'Reactant 2',
       'Product 1', 'Product 2']],Ga],axis=1, join='inner')
G_data.to_excel("9k_PredictedG1Dec.xlsx") 